<a href="https://colab.research.google.com/github/amircalabel/Test-with-ollama-for-tesis-project/blob/main/Test_with_ollama_for_tesis_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Conectando con drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Creando directorio en drive para los modelos

In [3]:
import os
import shutil

BASE_DIR = '/content/drive/MyDrive/rag_system'
os.makedirs(f'{BASE_DIR}/documents', exist_ok=True)   # Para los PDFs
os.makedirs(f'{BASE_DIR}/chroma_db', exist_ok=True)   # Para la base de datos vectorial
os.makedirs(f'{BASE_DIR}/models', exist_ok=True)      # Para modelos de embeddings

print(f"✅ Directorios creados en: {BASE_DIR}")
print("📁 Sube tus documentos PDF a la carpeta 'documents' en ese directorio de Drive")

✅ Directorios creados en: /content/drive/MyDrive/rag_system
📁 Sube tus documentos PDF a la carpeta 'documents' en ese directorio de Drive


Configurando ollama para usar drive

In [ ]:
os.environ['OLLAMA_MODELS'] = '/content/drive/MyDrive/ollama_models'

Instalacion de "zstd" para inst ollama

In [ ]:
! apt install zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 1 not upgraded.


Instalando Ollama

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


Iniciar el servidor en segundo plano

In [2]:
import subprocess, threading
def run_ollama():
    subprocess.run(["ollama", "serve"])
thread = threading.Thread(target=run_ollama)
thread.daemon = True
thread.start()

descarga de gemma 3

In [3]:
!ollama pull gemma3:4b

Testeando a Gemma 3

In [4]:
!ollama run gemma3:4b "Hola, estoy bien gracias por preguntar"

¡Me alegro mucho de oír eso! ¡Qué bueno que estás bien! 😊 ¿Hay algo en lo 
que te pueda ayudar o de lo que te gustaría hablar?




Instalando chromadb y dependencias

In [4]:
!pip install chromadb sentence-transformers pypdf2 pymupdf tiktoken

print("✅ Dependencias instaladas")

✅ Dependencias instaladas


In [1]:
import sys
!{sys.executable} -m pip install psutil


Configurando el sistema RAG

In [4]:
import chromadb
from chromadb.utils import embedding_functions
import fitz  # PyMuPDF para leer PDFs
import os
import re
from typing import List, Dict
import json


class RAGSystem:
    def __init__(self, persist_directory, model_name="all-MiniLM-L6-v2"):
        """
        Inicializa el sistema RAG con ChromaDB y un modelo de embeddings ligero
        """
        # Usar Sentence Transformers para embeddings
        self.embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
            model_name=model_name
        )

        # Conectar a ChromaDB con persistencia
        self.client = chromadb.PersistentClient(path=persist_directory)

        # Crear o obtener la colección de documentos
        self.collection = self.client.get_or_create_collection(
            name="documentos",
            embedding_function=self.embedding_fn
        )

        print(f"✅ Sistema RAG inicializado")
        print(f"   - Colección: {self.collection.name}")
        print(f"   - Documentos indexados: {self.collection.count()}")

    def extract_text_from_pdf(self, pdf_path: str) -> str:
        """Extrae texto de un archivo PDF"""
        try:
            doc = fitz.open(pdf_path)
            text = ""
            for page in doc:
                text += page.get_text()
            doc.close()
            return text
        except Exception as e:
            print(f"❌ Error leyendo {pdf_path}: {e}")
            return ""

    def chunk_text(self, text: str, chunk_size: int = 500, overlap: int = 100) -> List[str]:
        """
        Divide el texto en chunks más pequeños con solapamiento
        para mantener el contexto
        """
        # Limpiar texto
        text = re.sub(r'\s+', ' ', text).strip()

        chunks = []
        start = 0
        text_length = len(text)

        while start < text_length:
            end = min(start + chunk_size, text_length)

            # Si no es el final, buscar un punto o espacio para cortar bien
            if end < text_length:
                # Buscar el último punto o espacio antes de end
                last_period = text.rfind('.', start, end)
                last_space = text.rfind(' ', start, end)
                cut_point = max(last_period, last_space)
                if cut_point > start:
                    end = cut_point + 1

            chunk = text[start:end].strip()
            if chunk:
                chunks.append(chunk)

            start = end - overlap  # Solapamiento para mantener contexto

        return chunks

    def index_document(self, doc_path: str, doc_name: str = None, batch_size: int = 64):
        """Indexa un documento en ChromaDB, procesando en batches para optimizar memoria"""
        if doc_name is None:
            doc_name = os.path.basename(doc_path)

        print(f"📄 Procesando: {doc_name}")

        # Extraer texto
        text = self.extract_text_from_pdf(doc_path)
        if not text:
            print(f"   ⚠️ No se pudo extraer texto")
            return

        # Dividir en chunks
        chunks = self.chunk_text(text)
        print(f"   📝 {len(chunks)} chunks generados")

        if not chunks:
            print("   ⚠️ No se generaron chunks para indexar.")
            return

        # Procesar y agregar a ChromaDB en batches
        for i in range(0, len(chunks), batch_size):
            batch_chunks = chunks[i:i + batch_size]
            batch_ids = [f"{doc_name}_chunk_{j}" for j in range(i, i + len(batch_chunks))]
            batch_metadatas = [
                {"source": doc_name, "chunk_index": j, "total_chunks": len(chunks)}
                for j in range(i, i + len(batch_chunks))
            ]

            try:
                self.collection.add(
                    documents=batch_chunks,
                    metadatas=batch_metadatas,
                    ids=batch_ids
                )
                print(f"   ✅ Batch {i//batch_size + 1} de {len(chunks)//batch_size + 1} indexado ({len(batch_chunks)} chunks).")
            except Exception as e:
                print(f"   ❌ Error al indexar el batch {i//batch_size + 1}: {e}")
                # Consider logging the error and potentially continuing or stopping based on desired robustness

        print(f"   ✅ Indexación de '{doc_name}' completada.")

    def index_all_documents(self, documents_folder: str, batch_size: int = 32):
        """Indexa todos los PDFs en una carpeta"""
        pdf_files = [f for f in os.listdir(documents_folder) if f.lower().endswith('.pdf')]

        if not pdf_files:
            print(f"⚠️ No se encontraron PDFs en {documents_folder}")
            return

        print(f"📚 Encontrados {len(pdf_files)} documentos PDF")

        for pdf_file in pdf_files:
            pdf_path = os.path.join(documents_folder, pdf_file)
            self.index_document(pdf_path, pdf_file, batch_size=batch_size)

        print(f"\n🎉 Indexación completada. Total de chunks: {self.collection.count()}")

    def query(self, question: str, n_results: int = 5) -> Dict:
        """
        Busca los fragmentos más relevantes para una pregunta
        """
        results = self.collection.query(
            query_texts=[question],
            n_results=n_results
        )
        return results

    def get_context(self, question: str, n_results: int = 5) -> str:
        """
        Obtiene el contexto formateado para el modelo
        """
        results = self.query(question, n_results)

        if not results['documents'] or not results['documents'][0]:
            return "No se encontraron documentos relevantes."

        context_parts = []
        for i, (doc, metadata) in enumerate(zip(results['documents'][0], results['metadatas'][0]), 1):
            source = metadata['source']
            context_parts.append(f"[Fragmento {i} de {source}]\n{doc}\n")

        return "\n".join(context_parts)

# Inicializar el sistema
rag = RAGSystem(persist_directory=f'{BASE_DIR}/chroma_db')

print("\n✨ Sistema listo para usar")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Sistema RAG inicializado
   - Colección: documentos
   - Documentos indexados: 0

✨ Sistema listo para usar


In [5]:
import psutil

def get_memory_usage():
    process = psutil.Process(os.getpid())
    # Return memory in MB
    return process.memory_info().rss / (1024 * 1024)

# Re-initialize the system after modifications if needed (though it's usually automatic in a class modification)
# This is added here in case the user runs this cell directly after the modification
rag = RAGSystem(persist_directory=f'{BASE_DIR}/chroma_db')

print(f"Initial memory usage: {get_memory_usage():.2f} MB")


✅ Sistema RAG inicializado
   - Colección: documentos
   - Documentos indexados: 0
Initial memory usage: 741.31 MB


In [ ]:
rag.index_all_documents(f'{BASE_DIR}/documents')
print(f"Final memory usage after indexing: {get_memory_usage():.2f} MB")


Indexando Documentos

In [ ]:
rag.index_all_documents(f'{BASE_DIR}/documents')

📚 Encontrados 1 documentos PDF
📄 Procesando: Neuroarquitectura y materiales naturales. Su influencia en la sociedad actual. Arq. Rosalba Muñoz Rojas.pdf


Conectar con Ollama y hacer preguntas

In [ ]:
import subprocess
import threading
import time

class OllamaRAG:
    def __init__(self, rag_system, model_name="gemma3:4b"):
        self.rag = rag_system
        self.model = model_name
        self.ollama_ready = False

        # Verificar que Ollama está corriendo
        self._check_ollama()

    def _check_ollama(self):
        """Verifica si Ollama está disponible"""
        try:
            result = subprocess.run(["ollama", "list"], capture_output=True, text=True)
            if result.returncode == 0:
                print(f"✅ Ollama conectado correctamente")
                self.ollama_ready = True
            else:
                print("⚠️ Ollama no está respondiendo. Asegúrate de ejecutar: ollama serve")
        except FileNotFoundError:
            print("❌ Ollama no está instalado")

    def ask(self, question: str, n_results: int = 5) -> str:
        """
        Hace una pregunta sobre los documentos
        """
        if not self.ollama_ready:
            return "Error: Ollama no está disponible"

        # 1. Obtener contexto relevante
        context = self.rag.get_context(question, n_results)

        # 2. Construir el prompt
        prompt = f"""Eres un asistente experto en análisis de documentos. Responde la pregunta basándote ÚNICAMENTE en la información proporcionada en el contexto. Si la respuesta no está en el contexto, di claramente "No tengo información sobre eso en los documentos disponibles".

Contexto de los documentos:
{context}

Pregunta: {question}

Respuesta:"""

        # 3. Consultar a Ollama
        try:
            result = subprocess.run(
                ["ollama", "run", self.model, prompt],
                capture_output=True,
                text=True,
                timeout=120  # 2 minutos máximo
            )

            if result.returncode == 0:
                return result.stdout.strip()
            else:
                return f"Error en Ollama: {result.stderr}"
        except subprocess.TimeoutExpired:
            return "La consulta tomó demasiado tiempo. Intenta con una pregunta más específica."
        except Exception as e:
            return f"Error: {e}"

    def interactive_chat(self):
        """Modo interactivo para hacer preguntas"""
        print("\n" + "="*50)
        print("🤖 Modo interactivo RAG con Gemma 3")
        print("="*50)
        print("Comandos especiales:")
        print("  /salir - Terminar la conversación")
        print("  /stats - Mostrar estadísticas de la base de datos")
        print("="*50 + "\n")

        while True:
            question = input("\n📝 Tu pregunta: ").strip()

            if question.lower() in ['/salir', 'exit', 'quit']:
                print("👋 ¡Hasta luego!")
                break
            elif question.lower() == '/stats':
                print(f"\n📊 Estadísticas:")
                print(f"   - Documentos indexados: {rag.collection.count()} chunks")
                continue
            elif not question:
                continue

            print("\n🔍 Buscando respuesta...")
            answer = self.ask(question)
            print(f"\n💡 Respuesta:\n{answer}")
            print("\n" + "-"*50)

# Inicializar y usar
ollama_rag = OllamaRAG(rag)

# Ejemplo de pregunta única
# respuesta = ollama_rag.ask("¿Cuál es el tema principal del documento?")
# print(respuesta)

# O modo interactivo
ollama_rag.interactive_chat()

✅ Ollama conectado correctamente

🤖 Modo interactivo RAG con Gemma 3
Comandos especiales:
  /salir - Terminar la conversación
  /stats - Mostrar estadísticas de la base de datos


📝 Tu pregunta: que es neuroarquitectura

🔍 Buscando respuesta...

💡 Respuesta:
No tengo información sobre eso en los documentos disponibles.

--------------------------------------------------

📝 Tu pregunta: /salir
👋 ¡Hasta luego!
